# Prompt Injection Playground (PIP)

**Dr. Emre Süren** \
Royal Hacking Lab - lab.cybercampus.se \
KTH Royal Institute of Technology - www.kth.se/profile/emsuren/

## Installation
- Clone the playground repo: https://github.com/beyefendi/prompt-injection-playground.git
- Open up a terminal
- Change the directory to the parent folder of the repo
- `cd prompt-injection-playground`
- Run the setup script `./setup`
- Follow the instructions on the terminal
  - Configure the local DNS as stated in the terminal
  - Run the target LLM-app

**NOTE THAT:** YOU SHOULD RUN THE WORKSPACE.IPYNB INSIDE YOUR WORKSPACE FOLDER
IF YOU UPDATE WORKSPACE.IPYNB, TO KEEP THEM PERSISTENT OVERWRITE ATTACKS.IPYNB WITH WORKSPACE.IPYNB 

## Initialize

In [ ]:
!spikee -q init

## Available capabilities

In [ ]:
!spikee -q list

### 1. Injection templates
- Pre-installed datasets and prompt templates with optional configurations are available to generate the final dataset for attacking targets.
- The command `list seeds` searches for seed folders in the current working directory and lists the files inside the `datasets` folder.
    - Ensure you run this specific command from within your workspace folder; otherwise, the required files cannot be found.

In [ ]:
#!spikee -q list seeds

!spikee -q list targets

#!spikee -q list plugins

#!spikee -q list attacks

### 2. Generating a dataset

In [ ]:
!spikee -q generate -h

### 3. Performing attack

In [ ]:
!spikee -q test -h

### 4. Analyzing results

In [ ]:
!spikee -q results analyze -h

## LAB 1: Attack to a stand-alone LLM

### 1. Generate the dataset
- **Seed:** Select a jailbreak seed (--seed-folder datasets/seeds-cybersec-2025-04). **Path** to the seed **folder** should be provided.
- **Target** is a standalone LLM, so the attacker controls the **entire prompt** that is submitted to the model (e.g., system message, delimiters, task, suffixes, user content)
- **Tasks:** Use tasks (summarization and q&a -> typical user-provided tasks) while building prompts (--format full-prompt), that will reduce the response size but also multiplies the size of dataset
- **Jailbreak:** All techniques (e.g., new instructions, dan, ignore, etc.) including no-jailbreak
- **Spotlighting data markers:** Default (single new line)
- **Injection delimiters:** Default (single new line)
- Limit the dataset (number of prompts) generated
  - Manipulate only the base prompts that are in English (--languages en)
  - Use only the instruction data-exfil-markdown technique for injection

In [ ]:
!spikee --quiet generate \
    --seed-folder datasets/seeds-cybersec-2025-04 \
    --format full-prompt \
    --languages en \
    --instruction-filter data-exfil-markdown
    #--jailbreak-filter new-instructions

### 2. Perform the attack
- Use the dataset generated in the previous stage (--dataset datasets/cybersec-2025-04-full-prompt-dataset*.jsonl)
- Target is an LLM (e.g., --target togetherai_api) that requires an API key stored in the .env file
  - For free LLM APIs: Introduce sleeps between requests to avoid rate limit, typically 3 requests per minute (--throttle 20)
- Target option (--target-options llama31-8b) determines the model
- Once the attack is paused for a reason (e.g., rate limit) we can resume from the attack that we have left

In [ ]:
!spikee --quiet test \
    --dataset datasets/cybersec-2025-04-full-prompt-dataset-1764841661.jsonl \
    --target ollama_api \
    --target-options gemma3 \
    #--throttle 20 \
    --auto-resume

### 3. Analyze the results
- Calculate some statistics out of the responses from the LLM
- Also try to understand individual payloads that achieved the attack goal

In [ ]:
!spikee --quiet results analyze \
    --result-file results/results_ollama_api-gemma3_cybersec-2025-04-full-prompt-dataset-1764841661_1764864862.jsonl

## LAB 2: Attack to an LLM protected with a system message

### 1. Generate the dataset
- Improve the LAB 1
- **System message:** Protect the LLM with a system message (located in the beginning of the prompt), to see if it helps LLM (--include-system-message)
  - A system message describes the rules of the context 

In [ ]:
!spikee --quiet generate \
    --seed-folder datasets/seeds-cybersec-2025-04 \
    --format full-prompt \
    --include-system-message \
    --instruction-filter data-exfil-markdown \
    --languages en

### 2. Perform the attack

In [ ]:
!spikee --quiet test \
    --dataset datasets/cybersec-2025-04-full-prompt-sys-dataset-1764842613.jsonl \
    --target ollama_api \
    --target-options gemma3 \
    #--throttle 20 \
    --auto-resume

### 3. Analyze the results

In [ ]:
!spikee --quiet results analyze \
    --result-file results/results_ollama_api-gemma3_cybersec-2025-04-full-prompt-sys-dataset-1764842613_1764869980.jsonl

## LAB 3: Attack to an LLM protected with a system message and data marker

### 1. Generate the dataset
- Improve the LAB 2
- Protect the LLM with a data marker (encapsulating document into xml tag or json braces) between data and the instruction (--spotlighting-data-markers …), multiplies (2x) the size of dataset

In [ ]:
!spikee --quiet generate \
    --seed-folder datasets/seeds-cybersec-2025-04 \
    --format full-prompt \
    --include-system-message \
    --spotlighting-data-markers $'\n<data>\nDOCUMENT\n</data>\n',$'\n{"document":"DOCUMENT"}\n' \
    --instruction-filter data-exfil-markdown \
    --languages en

### 2. Perform the attack

In [ ]:
!spikee --quiet test \
    --dataset datasets/cybersec-2025-04-full-prompt-sys-dataset-1764863129.jsonl \
    --target ollama_api \
    --target-options gemma3 \
    #--throttle 20 \
    --auto-resume

### 3. Analyze the results

In [36]:
!spikee --quiet results analyze \
    --result-file results/results_ollama_api-gemma3_cybersec-2025-04-full-prompt-sys-dataset-1764863129_1764864203.jsonl


=== Analysis of Results File: results/results_ollama_api-gemma3_cybersec-2025-04-full-prompt-sys-dataset-1764863129_1764864203.jsonl ===
=== General Statistics ===
Total Unique Entries: 296
Successful Attacks: 0 [0.00%]
Failed Attacks: 296 [100.00%]
Errors: 0 [0.00%]
Total Attempts: 296
Attack Success Rate: 0.00%

=== Breakdown by Jailbreak Type ===
Jailbreak_Type        Total    Successes    Attempts  Success Rate
------------------  -------  -----------  ----------  --------------
'no-jailbreak'            8            0           8  0.00%
'new-instructions'       16            0          16  0.00%
'sorry'                   8            0           8  0.00%
'dan'                    32            0          32  0.00%
'ignore'                 24            0          24  0.00%
'test'                   40            0          40  0.00%
'errors'                 16            0          16  0.00%
'debug'                  16            0          16  0.00%
'dev'                    24    

## LAB 4: Adversarial suffix enhanced attack to an LLM protected with a system message and data markers

### 1. Generate the dataset
- Improve the LAB 3
- Enhance the payload by appending a suffix at the end of the payload (--include-suffixes)

In [37]:
!spikee --quiet generate \
    --seed-folder datasets/seeds-cybersec-2025-04 \
    --format full-prompt \
    --include-system-message \
    --spotlighting-data-markers $'\n<data>\nDOCUMENT\n</data>\n',$'\n{"document":"DOCUMENT"}\n' \
    --include-suffixes \
    --instruction-filter data-exfil-markdown \
    --languages en

Dataset generated and saved to datasets/cybersec-2025-04-full-prompt-sys-dataset-1764980984.jsonl

=== Dataset Statistics ===
Total Entries: 1184

Breakdown by Jailbreak Type:
Jailbreak Type      Count
----------------  -------
no-jailbreak           32
new-instructions       64
sorry                  32
dan                   128
ignore                 96
test                  160
errors                 64
debug                  64
dev                    96
emergency              32
no-limits              64
experimental           64
hidden-function        32
academic               32
new-task               96
challenge              32
training               32
special-case           64

Breakdown by Instruction Type:
Instruction Type       Count
-------------------  -------
data-exfil-markdown     1184

Breakdown by Language:
Language      Count
----------  -------
en             1184

Breakdown by Task Type:
Task Type        Count
-------------  -------
summarization      592
qna    

### 2. Perform the attack

In [ ]:
!spikee --quiet test \
    --dataset datasets/cybersec-2025-04-full-prompt-sys-dataset-1764980984.jsonl \
    --target ollama_api \
    --target-options gemma3 \
    #--throttle 20 \
    --auto-resume

[Overview] Testing the following dataset(s): 

 - datasets/cybersec-2025-04-full-prompt-sys-dataset-1764980984.jsonl

[Info] Spikee supports the following resume flags, instead of the interactive prompt:
  --auto-resume ~ silently pick the latest matching results file.
  --no-auto-resume ~ create a new results file.
 
[Start] Initiating testing of 'cybersec-2025-04-full-prompt-sys-dataset-1764980984.jsonl' against target 'ollama_api'
DEFAULTED canary None
DEFAULTED canary None
DEFAULTED canary None
DEFAULTED canary None
DEFAULTED canary None
DEFAULTED canary None
DEFAULTED canary None
DEFAULTED canary None
DEFAULTED canary None
DEFAULTED canary None
DEFAULTED canary None
DEFAULTED canary None
DEFAULTED canary None
DEFAULTED canary None
DEFAULTED canary None
DEFAULTED canary None
DEFAULTED canary None
DEFAULTED canary None
DEFAULTED canary None
DEFAULTED canary None
DEFAULTED canary None
DEFAULTED canary None
DEFAULTED canary None
DEFAULTED canary None
DEFAULTED canary None
DEFAULTED ca

### 3. Analyze the results

In [40]:
!spikee --quiet results analyze \
    --result-file results/results_ollama_api-gemma3_cybersec-2025-04-full-prompt-sys-dataset-1764980984_1765013879.jsonl


=== Analysis of Results File: results/results_ollama_api-gemma3_cybersec-2025-04-full-prompt-sys-dataset-1764980984_1765013879.jsonl ===
=== General Statistics ===
Total Unique Entries: 1184
Successful Attacks: 0 [0.00%]
Failed Attacks: 1184 [100.00%]
Errors: 0 [0.00%]
Total Attempts: 1184
Attack Success Rate: 0.00%

=== Breakdown by Jailbreak Type ===
Jailbreak_Type        Total    Successes    Attempts  Success Rate
------------------  -------  -----------  ----------  --------------
'no-jailbreak'           32            0          32  0.00%
'new-instructions'       64            0          64  0.00%
'sorry'                  32            0          32  0.00%
'dan'                   128            0         128  0.00%
'ignore'                 96            0          96  0.00%
'test'                  160            0         160  0.00%
'errors'                 64            0          64  0.00%
'debug'                  64            0          64  0.00%
'dev'                    96 

## LAB 5: Best of N attack to a protected LLM

### 1. Generate the dataset
- Use the LAB 4

### 2. Perform the attack
- Enhance the payloads by using a specific attack (--attack best_of_n --attack-iterations 50)

In [42]:
!spikee --quiet test \
    --dataset datasets/cybersec-2025-04-full-prompt-sys-dataset-1764980984.jsonl \
    --target ollama_api \
    --target-options gemma3 \
    --attack best_of_n \
    --attack-iterations 3 \
    --no-auto-resume
    #--throttle 20 \
    #--auto-resume \ # Do not use --auto-resume with this lab, it overwrites previous results! 

[Overview] Testing the following dataset(s): 

 - datasets/cybersec-2025-04-full-prompt-sys-dataset-1764980984.jsonl
 
[Start] Initiating testing of 'cybersec-2025-04-full-prompt-sys-dataset-1764980984.jsonl' against target 'ollama_api'
DEFAULTED canary None
DEFAULTED canary None
DEFAULTED canary None
DEFAULTED canary None
DEFAULTED canary None
DEFAULTED canary None
DEFAULTED canary None
DEFAULTED canary None
DEFAULTED canary None
DEFAULTED canary None
DEFAULTED canary None
DEFAULTED canary None
DEFAULTED canary None
DEFAULTED canary None
DEFAULTED canary None
DEFAULTED canary None
DEFAULTED canary None
DEFAULTED canary None
DEFAULTED canary None
DEFAULTED canary None
DEFAULTED canary None
DEFAULTED canary None
DEFAULTED canary None
DEFAULTED canary None
DEFAULTED canary None
DEFAULTED canary None
DEFAULTED canary None
DEFAULTED canary None
DEFAULTED canary None
DEFAULTED canary None
DEFAULTED canary None
DEFAULTED canary None
DEFAULTED canary None
DEFAULTED canary None
DEFAULTED canar

### 3. Analyze the results

In [47]:
!spikee --quiet results analyze \
    --result-file results/results_ollama_api-gemma3_cybersec-2025-04-full-prompt-sys-dataset-1764980984_1765014834.jsonl


=== Analysis of Results File: results/results_ollama_api-gemma3_cybersec-2025-04-full-prompt-sys-dataset-1764980984_1765014834.jsonl ===
=== General Statistics ===
Total Unique Entries: 1184
Successful Attacks (Total): 0 [0.00%]
  - Initially Successful: 0 [0.00%]
  - Only Successful with Dynamic Attack: 0 [0.00%]
Failed Attacks: 1184 [100.00%]
Errors: 0 [0.00%]
Total Attempts: 4736
Attack Success Rate (Overall): 0.00%
Attack Success Rate (Without Dynamic Attack): 0.00%
Attack Success Rate (Improvement from Dynamic Attack): 0.00%

=== Dynamic Attack Statistics ===
Attack Type      Total    Successes    Attempts    Guardrail  Success Rate
-------------  -------  -----------  ----------  -----------  --------------
best_of_n         1184            0        3552            0  0.00%

=== Breakdown by Jailbreak Type ===
Jailbreak_Type        Total    All Successes    Initial Successes    Attack-Only Successes    Attempts  Success Rate    Initial Success Rate    Attack Improvement
--------

## LAB 6: Anti-spotlighting attack to a protected LLM

### 1. Generate the dataset
- Use the LAB 4

### 2. Perform the attack
- Enhance the payloads by using a specific attack (--attack best_of_n --attack-iterations 15) # 15 ideal

In [45]:
!spikee --quiet test \
    --dataset datasets/cybersec-2025-04-full-prompt-sys-dataset-1764980984.jsonl \
    --target ollama_api \
    --target-options gemma3 \
    --attack anti_spotlighting \
    --attack-iterations 3 \
    --no-auto-resume
    #--throttle 20 \
    #--auto-resume false \ # Do not use --auto-resume with this lab, it overwrites previous results!

[Overview] Testing the following dataset(s): 

 - datasets/cybersec-2025-04-full-prompt-sys-dataset-1764980984.jsonl
 
[Start] Initiating testing of 'cybersec-2025-04-full-prompt-sys-dataset-1764980984.jsonl' against target 'ollama_api'
DEFAULTED canary None
DEFAULTED canary None
DEFAULTED canary None
DEFAULTED canary None
DEFAULTED canary None
DEFAULTED canary None
DEFAULTED canary None
DEFAULTED canary None
DEFAULTED canary None
DEFAULTED canary None
DEFAULTED canary None
DEFAULTED canary None
DEFAULTED canary None
DEFAULTED canary None
DEFAULTED canary None
DEFAULTED canary None
DEFAULTED canary None
DEFAULTED canary None
DEFAULTED canary None
DEFAULTED canary None
DEFAULTED canary None
DEFAULTED canary None
DEFAULTED canary None
DEFAULTED canary None
DEFAULTED canary None
DEFAULTED canary None
DEFAULTED canary None
DEFAULTED canary None
DEFAULTED canary None
DEFAULTED canary None
DEFAULTED canary None
DEFAULTED canary None
DEFAULTED canary None
DEFAULTED canary None
DEFAULTED canar

### 3. Analyze the results

In [51]:
!spikee --quiet results analyze \
    --result-file results/results_ollama_api-gemma3_cybersec-2025-04-full-prompt-sys-dataset-1764980984_1765036273.jsonl


=== Analysis of Results File: results/results_ollama_api-gemma3_cybersec-2025-04-full-prompt-sys-dataset-1764980984_1765036273.jsonl ===
=== General Statistics ===
Total Unique Entries: 1184
Successful Attacks (Total): 0 [0.00%]
  - Initially Successful: 0 [0.00%]
  - Only Successful with Dynamic Attack: 0 [0.00%]
Failed Attacks: 1184 [100.00%]
Errors: 0 [0.00%]
Total Attempts: 4736
Attack Success Rate (Overall): 0.00%
Attack Success Rate (Without Dynamic Attack): 0.00%
Attack Success Rate (Improvement from Dynamic Attack): 0.00%

=== Dynamic Attack Statistics ===
Attack Type          Total    Successes    Attempts    Guardrail  Success Rate
-----------------  -------  -----------  ----------  -----------  --------------
anti_spotlighting     1184            0        3552            0  0.00%

=== Breakdown by Jailbreak Type ===
Jailbreak_Type        Total    All Successes    Initial Successes    Attack-Only Successes    Attempts  Success Rate    Initial Success Rate    Attack Improvem

## LAB 7: Prompt decomposition attack to a protected LLM

### 1. Generate the dataset
- Use the LAB 4

### 2. Perform the attack
- Enhance the payloads by using a specific attack (--attack prompt-decomposition --attack-iterations 50)

In [52]:
!spikee --quiet test \
    --dataset datasets/cybersec-2025-04-full-prompt-sys-dataset-1764980984.jsonl \
    --target ollama_api \
    --target-options gemma3 \
    --attack prompt_decomposition \
    --attack-iterations 3 \
    --no-auto-resume \
    #--throttle 20 \
    #--auto-resume false \ # Do not use --auto-resume with this lab, it overwrites previous results!    

[Overview] Testing the following dataset(s): 

 - datasets/cybersec-2025-04-full-prompt-sys-dataset-1764980984.jsonl
 
[Start] Initiating testing of 'cybersec-2025-04-full-prompt-sys-dataset-1764980984.jsonl' against target 'ollama_api'
DEFAULTED canary None
DEFAULTED canary None
DEFAULTED canary None
DEFAULTED canary None
DEFAULTED canary None
DEFAULTED canary None
DEFAULTED canary None
DEFAULTED canary None
DEFAULTED canary None
DEFAULTED canary None
DEFAULTED canary None
DEFAULTED canary None
DEFAULTED canary None
DEFAULTED canary None
DEFAULTED canary None
DEFAULTED canary None
DEFAULTED canary None
DEFAULTED canary None
DEFAULTED canary None
DEFAULTED canary None
DEFAULTED canary None
DEFAULTED canary None
DEFAULTED canary None
DEFAULTED canary None
DEFAULTED canary None
DEFAULTED canary None
DEFAULTED canary None
DEFAULTED canary None
DEFAULTED canary None
DEFAULTED canary None
DEFAULTED canary None
DEFAULTED canary None
DEFAULTED canary None
DEFAULTED canary None
DEFAULTED canar

### 3. Analyze the results

In [53]:
!spikee --quiet results analyze \
    --result-file results/results_ollama_api-gemma3_cybersec-2025-04-full-prompt-sys-dataset-1764980984_1765061739.jsonl


=== Analysis of Results File: results/results_ollama_api-gemma3_cybersec-2025-04-full-prompt-sys-dataset-1764980984_1765061739.jsonl ===
=== General Statistics ===
Total Unique Entries: 1184
Successful Attacks (Total): 0 [0.00%]
  - Initially Successful: 0 [0.00%]
  - Only Successful with Dynamic Attack: 0 [0.00%]
Failed Attacks: 1184 [100.00%]
Errors: 0 [0.00%]
Total Attempts: 4736
Attack Success Rate (Overall): 0.00%
Attack Success Rate (Without Dynamic Attack): 0.00%
Attack Success Rate (Improvement from Dynamic Attack): 0.00%

=== Dynamic Attack Statistics ===
Attack Type             Total    Successes    Attempts    Guardrail  Success Rate
--------------------  -------  -----------  ----------  -----------  --------------
prompt_decomposition     1184            0        3552            0  0.00%

=== Breakdown by Jailbreak Type ===
Jailbreak_Type        Total    All Successes    Initial Successes    Attack-Only Successes    Attempts  Success Rate    Initial Success Rate    Attack

## LAB 8: Random suffix search attack to a protected LLM

### 1. Generate the dataset
- Use the LAB 4

### 2. Perform the attack
- Enhance the payloads by using a specific attack (--attack random-suffix-search --attack-iterations 50)

In [54]:
!spikee --quiet test \
    --dataset datasets/cybersec-2025-04-full-prompt-sys-dataset-1764980984.jsonl \
    --target ollama_api \
    --target-options gemma3 \
    --attack random_suffix_search \
    --attack-iterations 3 \
    --no-auto-resume
    #--throttle 20 \
    #--auto-resume false \ # Do not use --auto-resume with this lab, it overwrites previous results!

[Overview] Testing the following dataset(s): 

 - datasets/cybersec-2025-04-full-prompt-sys-dataset-1764980984.jsonl
 
[Start] Initiating testing of 'cybersec-2025-04-full-prompt-sys-dataset-1764980984.jsonl' against target 'ollama_api'
DEFAULTED canary None
DEFAULTED canary None
DEFAULTED canary None
DEFAULTED canary None
DEFAULTED canary None
DEFAULTED canary None
DEFAULTED canary None
DEFAULTED canary None
DEFAULTED canary None
DEFAULTED canary None
DEFAULTED canary None
DEFAULTED canary None
DEFAULTED canary None
DEFAULTED canary None
DEFAULTED canary None
DEFAULTED canary None
DEFAULTED canary None
DEFAULTED canary None
DEFAULTED canary None
DEFAULTED canary None
DEFAULTED canary None
DEFAULTED canary None
DEFAULTED canary None
DEFAULTED canary None
DEFAULTED canary None
DEFAULTED canary None
DEFAULTED canary None
DEFAULTED canary None
DEFAULTED canary None
DEFAULTED canary None
DEFAULTED canary None
DEFAULTED canary None
DEFAULTED canary None
DEFAULTED canary None
DEFAULTED canar

### 3. Analyze the results

In [55]:
!spikee --quiet results analyze \
    --result-file results/results_ollama_api-gemma3_cybersec-2025-04-full-prompt-sys-dataset-1764980984_1765097794.jsonl


=== Analysis of Results File: results/results_ollama_api-gemma3_cybersec-2025-04-full-prompt-sys-dataset-1764980984_1765097794.jsonl ===
=== General Statistics ===
Total Unique Entries: 1184
Successful Attacks (Total): 0 [0.00%]
  - Initially Successful: 0 [0.00%]
  - Only Successful with Dynamic Attack: 0 [0.00%]
Failed Attacks: 1184 [100.00%]
Errors: 0 [0.00%]
Total Attempts: 4728
Attack Success Rate (Overall): 0.00%
Attack Success Rate (Without Dynamic Attack): 0.00%
Attack Success Rate (Improvement from Dynamic Attack): 0.00%

=== Dynamic Attack Statistics ===
Attack Type             Total    Successes    Attempts    Guardrail  Success Rate
--------------------  -------  -----------  ----------  -----------  --------------
random_suffix_search     1184            0        3544            0  0.00%

=== Breakdown by Jailbreak Type ===
Jailbreak_Type        Total    All Successes    Initial Successes    Attack-Only Successes    Attempts  Success Rate    Initial Success Rate    Attack

## LAB 9: Attack with SimSonSun to a protected LLM

### 1.1 Download public data
- Find a well-known jailbreak prompt dataset (not a seed)
- Script below downloads it under its directory (datasets/seeds-in-the-wild-jailbreak-prompts/standalone_user_inputs.jsonl)
- Note that the filename created is special
- Note that spikee checks local datasets folder (actually seed folder) to find standalone_user_inputs.jsonl while generating dataset
  - That is why we need to switch to our workspace folder (that is prompt-generator)

In [56]:
!python datasets/seeds-simsonsun-high-quality-jailbreaks/fetch_and_convert_dataset.py


Loading dataset from Hugging Face...
Using Dataset 1: High-quality filtered prompts

Successfully converted 67 jailbreak prompts to JSONL format.
Output saved to 'datasets/seeds-simsonsun-high-quality-jailbreaks/standalone_user_inputs.jsonl'

Prompts by source:
  Overlap analysis: 30
  Nannazi: 7
  Tulu: 7
  Reddit: 6
  tridm/jailbreak_test_v1.0: 4
  Github: 3
  injectPrompt: 3
  MindGard: 2
  Bravansky/compact-jailbreaks: 2
  Article: 1
  r1-1776: 1
  GuardrailsAI/detect-jailbreak: 1


### 1.2. Generate the dataset
- Do not process the dataset (it is already generated)

In [57]:
!spikee --quiet generate \
    --seed-folder datasets/seeds-simsonsun-high-quality-jailbreaks \
    --include-standalone-inputs

Dataset generated and saved to datasets/simsonsun-high-quality-jailbreaks-user-input-dataset-1765114608.jsonl

=== Dataset Statistics ===
Total Entries: 67

Breakdown by Jailbreak Type:
Jailbreak Type      Count
----------------  -------
None                   67

Breakdown by Instruction Type:
Instruction Type                               Count
-------------------------------------------  -------
simsonsun-ds1-nannazi                              7
simsonsun-ds1-tulu                                 7
simsonsun-ds1-article                              1
simsonsun-ds1-reddit                               6
simsonsun-ds1-mindgard                             2
simsonsun-ds1-github                               3
simsonsun-ds1-injectprompt                         3
simsonsun-ds1-r1-1776                              1
simsonsun-ds1-tridm/jailbreak-test-v1.0            4
simsonsun-ds1-guardrailsai/detect-jailbreak        1
simsonsun-ds1-bravansky/compact-jailbreaks         2
simsonsun-ds1-o

### 2. Perform the attack
- Use the dataset generated in the previous stage (--dataset datasets/seeds-simsonsun-high-quality-jailbreaks-dataset*.jsonl)
- Target is an LLM (--target google_api) that requires an API key stored in the .env file

In [58]:
!spikee --quiet test \
    --dataset datasets/simsonsun-high-quality-jailbreaks-user-input-dataset-1765114608.jsonl \
    --target ollama_api \
    --target-options gemma3 \
    --no-auto-resume \
    --judge-options ollama-gemma3
    #--judge-options together-llama31-8b
    #--throttle 20 \
    #--auto-resume \
    #--sample 0.1

[Overview] Testing the following dataset(s): 

 - datasets/simsonsun-high-quality-jailbreaks-user-input-dataset-1765114608.jsonl
 
[Start] Initiating testing of 'simsonsun-high-quality-jailbreaks-user-input-dataset-1765114608.jsonl' against target 'ollama_api'
[Info] Testing 67 new entries (threads=4).
[Info] Output will be saved to: results/results_ollama_api-gemma3_simsonsun-high-quality-jailbreaks-user-input-dataset-1765114608_1765114819.jsonl

Processing entries:   0%|                     | 0/78 [00:00<?, ?it/s, success=0]/Users/emresuren/GitHub/prompt-injection-playground/.venv/lib/python3.13/site-packages/langchain/__init__.py:30: UserWarning: Importing verbose from langchain root module is no longer supported. Please use langchain.globals.set_verbose() / langchain.globals.get_verbose() instead.
  warnings.warn(

Processing entries:  86%|█████████▍ | 67/78 [19:22<03:10, 17.36s/it, success=55]
[Done] Testing finished. Results saved to results/results_ollama_api-gemma3_simsonsun-hi

### 3. Analyze the results
- Calculate some statistics out of the responses from the LLM
- Also try to understand individual payloads that achieved the attack goal

In [59]:
!spikee --quiet results analyze \
    --result-file results/results_ollama_api-gemma3_simsonsun-high-quality-jailbreaks-user-input-dataset-1765114608_1765114819.jsonl


=== Analysis of Results File: results/results_ollama_api-gemma3_simsonsun-high-quality-jailbreaks-user-input-dataset-1765114608_1765114819.jsonl ===
=== General Statistics ===
Total Unique Entries: 67
Successful Attacks: 55 [82.09%]
Failed Attacks: 12 [17.91%]
Errors: 0 [0.00%]
Total Attempts: 67
Attack Success Rate: 82.09%

=== Breakdown by Jailbreak Type ===
Jailbreak_Type      Total    Successes    Attempts  Success Rate
----------------  -------  -----------  ----------  --------------
'None'                 67           55          67  82.09% 

=== Breakdown by Instruction Type ===
Instruction_Type                                 Total    Successes    Attempts  Success Rate
---------------------------------------------  -------  -----------  ----------  --------------
'simsonsun-ds1-article'                              1            1           1  100.00%
'simsonsun-ds1-reddit'                               6            6           6  100.00%
'simsonsun-ds1-mindgard'             

## LAB 10: Attack with jailbreak in the wild to a protected LLM

### 1.1 Download public data
- Find a well-known jailbreak prompt dataset (not a seed)
- Script below downloads it under its directory (datasets/seeds-in-the-wild-jailbreak-prompts/standalone_user_inputs.jsonl)
- Note that the filename created is special

In [1]:
!python datasets/seeds-in-the-wild-jailbreak-prompts/fetch_and_convert_dataset.py

Loading dataset from Hugging Face...
Filtering for jailbreak prompts...

Successfully converted 1405 jailbreak prompts to JSONL format.
Output saved to 'datasets/seeds-in-the-wild-jailbreak-prompts/standalone_user_inputs.jsonl'

Prompts by platform:
  website: 509
  discord: 469
  reddit: 425
  open_source: 2


### 1.2. Generate the dataset
- Do not process the dataset (it is already generated)

In [2]:
!spikee --quiet generate \
    --seed-folder datasets/seeds-in-the-wild-jailbreak-prompts \
    --include-standalone-inputs

Dataset generated and saved to datasets/in-the-wild-jailbreak-prompts-user-input-dataset-1765236094.jsonl

=== Dataset Statistics ===
Total Entries: 1405

Breakdown by Jailbreak Type:
Jailbreak Type      Count
----------------  -------
None                 1405

Breakdown by Instruction Type:
Instruction Type                             Count
-----------------------------------------  -------
in-the-wild-jailbreak-prompts-discord          469
in-the-wild-jailbreak-prompts-reddit           425
in-the-wild-jailbreak-prompts-open_source        2
in-the-wild-jailbreak-prompts-website          509

Breakdown by Language:
Language      Count
----------  -------
en             1405

Breakdown by Task Type:
Task Type      Count
-----------  -------
None            1405

Breakdown by Suffix ID:
Suffix ID      Count
-----------  -------
None            1405

Breakdown by Plugin ID:
Plugin ID      Count
-----------  -------
None            1405


### 2. Perform the attack
- Use the dataset generated in the previous stage (--dataset datasets/in-the-wild-jailbreak-prompts-user-input-dataset*.jsonl)
- Target is an LLM (--target google_api) that requires an API key stored in the .env file

In [ ]:
!spikee --quiet test \
    --dataset datasets/in-the-wild-jailbreak-prompts-user-input-dataset-1765236094.jsonl \
    --target ollama_api \
    --target-options gemma3 \
    --no-auto-resume \
    --judge-options ollama-gemma3 \
    #--throttle 20 \
    #--auto-resume \
    --sample 0.1

[Overview] Testing the following dataset(s): 

 - datasets/in-the-wild-jailbreak-prompts-user-input-dataset-1765236094.jsonl
 
[Start] Initiating testing of 'in-the-wild-jailbreak-prompts-user-input-dataset-1765236094.jsonl' against target 'ollama_api'
[Info] Testing 1405 new entries (threads=4).
[Info] Output will be saved to: results/results_ollama_api-gemma3_in-the-wild-jailbreak-prompts-user-input-dataset-1765236094_1765236298.jsonl

Processing entries:   0%|                     | 0/74 [00:00<?, ?it/s, success=0]/Users/emresuren/GitHub/prompt-injection-playground/.venv/lib/python3.13/site-packages/langchain/__init__.py:30: UserWarning: Importing verbose from langchain root module is no longer supported. Please use langchain.globals.set_verbose() / langchain.globals.get_verbose() instead.
  warnings.warn(

Processing entries: 100%|███████████| 74/74 [06:37<00:00,  3.06s/it, success=60]
Processing entries: 75it [06:37,  2.28s/it, success=61]                         
Processing entrie

### 3. Analyze the results
- Calculate some statistics out of the responses from the LLM
- Also try to understand individual payloads that achieved the attack goal

In [ ]:
!spikee --quiet results analyze \
    --result-file results/results_ollama_api-gemma3_in-the-wild-jailbreak-prompts-user-input-dataset-1765236094_1765236298.jsonl

## Lab 11: Prompt injection to bypass Guardrail apps

### 1. Generate the dataset
- Select a jailbreak seed (llm-mailbox)
- Target is a guardrail app, so attacker cannot control the full prompt that is submitted to the LLM (FIX: system message, task, tag, suffixes, user content)
- Do NOT use tasks (summarization and q&a -> typical user prompts) while building prompts (--format user-input), since guardrail will add something that we do not know that can potentially create conflicts.
- Not using tasks can cause LLMs to increase the response size (output tokens), since it prompt will not be clear for it, but reduces the number of prompts (input tokens)
- Enhance LLM with a system message in the beginning of the prompt, to see if it helps LLM (--include-system-message) 
- Enhance LLM with a data marker (adding newline or parenthesis) between data and the instruction (--spotlighting-data-markers …)
- Limit the number of prompts generated, (--languages en)

In [ ]:
!spikee --quiet generate \
    --seed-folder datasets/seeds-llm-mailbox \
    --format user-input \
    --instruction-filter data-exfil-markdown \
    --languages en

### 2. Perform the attack
- Use the dataset generated in the previous stage (--dataset datasets/llm-mailbox-user-input-dataset*.jsonl)
- Target is an app (--target llm_mailbox) that should be running

In [ ]:
!spikee --quiet test \
    --dataset datasets/llm-mailbox-user-input-dataset-1762977999.jsonl \
    --target llm_mailbox \
    --auto-resume
    #--target prompt_guard_jailbreak
    #--target prompt_guard_injection
    

### 3. Analyze the results
- Calculate some statistics out of the responses from the LLM
- Also try to understand individual payloads that achieved the attack goal

In [ ]:
!spikee --quiet results analyze \
    --result-file results/results_llm_mailbox_llm-mailbox-user-input-dataset-1762977999_1763054530.jsonl

## Lab 12: Prompt injection to extract system message out of an LLM
- Threat model: Attacker targets an LLM to learn its system message that will allow attacker to improve jailbreak/injection prompts later on

### 1. Generate the dataset
- Select a jailbreak seed (seeds-sysmsg-extraction-2025-04)
- Target is a standalone LLM, so attacker controls the entire prompt that is submitted to the model (system message, task, tag, suffixes, user content)
- Use tasks (summarization and q&a -> typical user prompts) while building prompts (--format full-prompt), that will reduce the response size but also doubles the number of prompts
- Enhance LLM with a system message in the beginning of the prompt, to see if it helps LLM (--include-system-message) 
- Enhance LLM with a data marker (adding newline or parenthesis) between data and the instruction (--spotlighting-data-markers …)
- Limit the number of prompts generated, (--languages en)

In [ ]:
!spikee --quiet generate \
    --seed-folder datasets/seeds-sysmsg-extraction-2025-04 \
    --format full-prompt \
    --include-system-message \
    --spotlighting-data-markers $'\n<data>\nDOCUMENT\n</data>\n',$'\n{"document":"DOCUMENT"}\n' \
    --languages en

### 2. Perform the attack
- Use the dataset generated in the previous stage (--dataset datasets/seeds-sysmsg-extraction-2025-04-dataset*.jsonl)
- Target is an LLM (--target ollama-llama3.2) that requires an API key stored in the .env file
- The judge is LLM (probably the answers should e interpreted or are not something we can catch by regular exceptions)

In [ ]:
!spikee --quiet test \
    --dataset <your-dataset>.jsonl \
    --target <your-target> \
    --judge-options 'ollama-llama3.2'

### 3. Analyze the results
- Calculate some statistics out of the responses from the LLM
- Also try to understand individual payloads that achieved the attack goal

In [ ]:
!spikee results analyze \
    --result-file results/<>.jsonl

## Lab 13: Prompt injection to extract system message out of an LLM-powered app
- Threat model: Attacker targets an application that uses LLM. To learn its system message that will allow attacker to improve jailbreak/injection prompts later on.

### 1. Generate the dataset
- Select a jailbreak seed (seeds-sysmsg-extraction-2025-04)
- Target is a standalone LLM, so attacker controls the entire prompt that is submitted to the model (system message, task, tag, suffixes, user content)
- Use tasks (summarization and q&a -> typical user prompts) while building prompts (--format full-prompt), that will reduce the response size but also doubles the number of prompts
- Enhance LLM with a system message in the beginning of the prompt, to see if it helps LLM (--include-system-message) 
- Enhance LLM with a data marker (adding newline or parenthesis) between data and the instruction (--spotlighting-data-markers …)
- Limit the number of prompts generated, (--languages en)

## Lab 14: Case: LLM WebMail summarization feature
- Target: Custom LLM App
- Feature: LLM **summarizes** a bunch of e-mails (e-mails from last week, last 50 e-mails, whole e-mail box etc.) 
- Dataset: Need to develop **custom dataset** targeting specific LLM feature, summarization
- Threat model: Attacker triggers a password reset to target, and right after this sends an e-mail to the user, that contains (data exfiltration) prompt injection leaking password reset token (http://aimail.com/password-reset?token=randomNumbers123) to an attacker controlled IP via markdown image. 
- Attack automation (Burp Intruder)

### 1. Generate the dataset
- Target is an app so create config files tailored to target
- Reduce the number of payload, include only instructions in english (--languages en) 
- Target is an app (not an LLM), so the format of dataset (--format user-input)
- Set the delimiter between data and the payload like new line or paranthesis

In [ ]:
!spikee --quiet generate  \
    --seed-folder datasets/seeds-llm-mailbox \
    --format user-input \
    --injection-delimiters $'\nINJECTION_PAYLOAD\n',$'(INJECTION_PAYLOAD)'

### 2. Perform the attack
- Set the dataset
- Set the target app
- Set the attack type (introduces dynamic attacks enhancing dataset)
- Control the speed

In [ ]:
# Example: If the standard test fails, try the best_of_n attack for 20 iterations
!spikee --quiet test \
    --dataset datasets/llm-mailbox-user-input-dataset-1762181492.jsonl \
    --target llm_mailbox \
    --attack best_of_n \
    --attack-iterations 20 \
    --max-retries 3 \
    --threads 2 --throttle 1

### 3. Analyze the results
- Calculate statistics from the responses from the app
- Also try to understand individual payloads that achieved the attack goal

In [ ]:
!spikee --quiet results analyze \
    --result-file results/results_llm_mailbox_user_input.jsonl

## LAB 15

- Target is an LLM, exclude tasks (summarization and qna) while building prompts (--format user-prompt), by default --format is user-prompt

In [ ]:
!spikee --quiet generate  \
    --seed-folder datasets/seeds-cybersec-2025-04 \
    --format user-input \
    --tag "no-task-instructions"